# Day 4 — Daily Returns + Distribution Validation + CAGR + Comparison Table

This notebook performs all Day-4 required computations:

1. **Daily returns** for all schemes using:
   `daily_return = nav_t / nav_{t-1} - 1`
2. **Validate** that the daily-return distribution looks reasonable (summary stats, histogram, outlier scan).
3. **Compute CAGR** for horizons **1yr, 3yr, 5yr** using:
   `CAGR = (NAV_end / NAV_start)^(1/n) - 1`
4. **Build comparison table** across all funds.

In [1]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd

import plotly.express as px
import plotly.graph_objects as go

In [2]:
# --- Repo-root detection (so notebook works from any working directory) ---

_HERE = Path(__file__).resolve() if '__file__' in globals() else Path.cwd()


def _find_repo_root(start: Path) -> Path:
    cand = start
    for _ in range(12):
        if (cand / 'Data' / 'processed' / 'nav_history_clean.csv').exists() and (cand / 'Data' / 'processed' / 'fund_master_clean.csv').exists():
            return cand
        if cand.name == 'notebooks':
            parent = cand.parent
            if (parent / 'Data' / 'processed' / 'nav_history_clean.csv').exists() and (parent / 'Data' / 'processed' / 'fund_master_clean.csv').exists():
                return parent
        cand = cand.parent
    return start.parent


_REPO_ROOT = _find_repo_root(_HERE)
DATA_DIR = _REPO_ROOT / 'Data' / 'processed'

nav_path = DATA_DIR / 'nav_history_clean.csv'
fund_path = DATA_DIR / 'fund_master_clean.csv'

if not nav_path.exists():
    raise FileNotFoundError(f'Missing file: {nav_path.resolve()}')
if not fund_path.exists():
    raise FileNotFoundError(f'Missing file: {fund_path.resolve()}')

print('DATA_DIR:', DATA_DIR.resolve())

DATA_DIR: C:\Mutual Fund Analytics\Data\processed


In [3]:
nav_df = pd.read_csv(nav_path)
fund_df = pd.read_csv(fund_path)

nav_df['date'] = pd.to_datetime(nav_df['date'], errors='coerce')
nav_df['amfi_code'] = pd.to_numeric(nav_df['amfi_code'], errors='coerce').astype('Int64')
nav_df['nav'] = pd.to_numeric(nav_df['nav'], errors='coerce')

fund_df['amfi_code'] = pd.to_numeric(fund_df['amfi_code'], errors='coerce').astype('Int64')

# Keep only rows with valid NAV and scheme mapping
nav_df = nav_df.dropna(subset=['amfi_code', 'date', 'nav']).copy()
fund_df = fund_df.dropna(subset=['amfi_code', 'scheme_name']).copy()

fund_df = fund_df.sort_values('amfi_code')
amfi_to_name = dict(zip(fund_df['amfi_code'].astype(int), fund_df['scheme_name']))
codes_sorted = fund_df['amfi_code'].astype(int).tolist()

print('Loaded schemes:', len(codes_sorted))
print('NAV rows:', len(nav_df))
print('NAV date range:', nav_df['date'].min(), '->', nav_df['date'].max())

Loaded schemes: 40
NAV rows: 64320
NAV date range: 2022-01-03 00:00:00 -> 2026-05-29 00:00:00


In [4]:
# ============================================================
# 1) DAILY RETURNS
# daily_return = nav_t / nav_{t-1} - 1
# ============================================================

nav_df = nav_df.sort_values(['amfi_code', 'date']).copy()

nav_df['daily_return'] = nav_df.groupby('amfi_code')['nav'].pct_change()
nav_df = nav_df.replace([np.inf, -np.inf], np.nan).copy()

nav_df['scheme_name'] = nav_df['amfi_code'].map(amfi_to_name)

daily_returns_df = nav_df.dropna(subset=['daily_return']).copy()

print('Daily returns rows:', len(daily_returns_df))
print('Daily return range (decimal):', float(daily_returns_df['daily_return'].min()), '-|>', float(daily_returns_df['daily_return'].max()))

Daily returns rows: 64280
Daily return range (decimal): -0.058102013949189124 -|> 0.06471309359097144


In [5]:
# Save for downstream (matches earlier Day-4 behavior)
out_path = DATA_DIR / 'daily_returns_all_schemes.csv'
cols = ['date', 'amfi_code', 'scheme_name', 'nav', 'daily_return']

daily_returns_df[cols].to_csv(out_path, index=False)

print('Wrote:', out_path.resolve())

Wrote: C:\Mutual Fund Analytics\Data\processed\daily_returns_all_schemes.csv


## Distribution validation

In [6]:
def describe_series(s: pd.Series) -> dict:
    s = s.dropna().astype(float)
    return {
        'count': int(s.shape[0]),
        'mean': float(s.mean()),
        'std': float(s.std(ddof=1)),
        'min': float(s.min()),
        'p01': float(np.quantile(s, 0.01)),
        'p05': float(np.quantile(s, 0.05)),
        'median': float(np.quantile(s, 0.50)),
        'p95': float(np.quantile(s, 0.95)),
        'p99': float(np.quantile(s, 0.99)),
        'max': float(s.max()),
    }


all_desc = describe_series(daily_returns_df['daily_return'])
print('All schemes daily_return summary (decimal; 0.01 = 1%)')
for k, v in all_desc.items():
    print(f'{k:>6}: {v}')

All schemes daily_return summary (decimal; 0.01 = 1%)
 count: 64280
  mean: 0.0004512000442774934
   std: 0.008705951210937613
   min: -0.058102013949189124
   p01: -0.023780554542765252
   p05: -0.014101990792956421
median: 0.0
   p95: 0.015711077438477685
   p99: 0.025741532499509025
   max: 0.06471309359097144


In [7]:
# Histogram (all schemes)
fig = px.histogram(
    daily_returns_df,
    x='daily_return',
    nbins=250,
    title='Distribution of Daily Returns (All Schemes)',
    labels={'daily_return': 'daily_return (decimal)'}
)
fig.update_layout(template='plotly_white')
fig.show()

In [8]:
# Outlier scan: unusually large daily moves
POS_THRESHOLD = 0.10   # +10%
NEG_THRESHOLD = -0.10  # -10%

mask_outliers = (daily_returns_df['daily_return'] > POS_THRESHOLD) | (daily_returns_df['daily_return'] < NEG_THRESHOLD)
outliers = daily_returns_df.loc[mask_outliers].copy()

print('Outliers count (abs move > 10%):', len(outliers))
if len(outliers) > 0:
    display(outliers.sort_values('daily_return').head(10)[['date', 'amfi_code', 'scheme_name', 'daily_return', 'nav']])

Outliers count (abs move > 10%): 0


In [9]:
# Per-scheme sanity: how many outlier days each scheme has
mask_outliers = (daily_returns_df['daily_return'] > POS_THRESHOLD) | (daily_returns_df['daily_return'] < NEG_THRESHOLD)

per_scheme = (
    daily_returns_df.assign(is_outlier=mask_outliers)
    .groupby(['amfi_code', 'scheme_name'])
    .agg(
        count=('daily_return', 'size'),
        mean=('daily_return', 'mean'),
        std=('daily_return', 'std'),
        outlier_days_abs_gt_10pct=('is_outlier', 'sum'),
        min_return=('daily_return', 'min'),
        max_return=('daily_return', 'max'),
    )
    .reset_index()
)

print('Top 10 schemes by #outlier days (abs>10%):')
display(per_scheme.sort_values('outlier_days_abs_gt_10pct', ascending=False).head(10))

Top 10 schemes by #outlier days (abs>10%):


,amfi_code,scheme_name,count,mean,std,outlier_days_abs_gt_10pct,min_return,max_return
0,100016,HDFC Top 100 Fund - Regular Plan - Growth,1607,0.000101,0.007749,0,-0.024744,0.032145
1,100025,HDFC Short Term Debt Fund - Regular - Growth,1607,0.000122,0.002081,0,-0.008188,0.008837
2,100033,HDFC Mid-Cap Opportunities Fund - Regular - Gr...,1607,0.000772,0.010097,0,-0.044238,0.041954
3,101206,ABSL Frontline Equity Fund - Regular - Growth,1607,0.000609,0.007768,0,-0.038121,0.033956
4,101207,ABSL Small Cap Fund - Regular - Growth,1607,0.000303,0.013741,0,-0.051847,0.054851
5,101208,ABSL Liquid Fund - Regular - Growth,1607,0.000173,0.000291,0,-0.000766,0.001247
6,102885,UTI Nifty 50 Index Fund - Regular - Growth,1607,0.000482,0.006844,0,-0.021104,0.025565
7,102886,UTI Mid Cap Fund - Regular - Growth,1607,0.000079,0.009659,0,-0.036695,0.040745
8,102887,UTI Flexi Cap Fund - Regular - Growth,1607,0.000461,0.008398,0,-0.031271,0.032386
9,118632,Nippon India Large Cap Fund - Regular - Growth,1607,0.000619,0.007545,0,-0.028490,0.028218


In [10]:
# Plot an example scheme daily returns time series
example_code = codes_sorted[0]
ex = daily_returns_df.loc[daily_returns_df['amfi_code'] == example_code].sort_values('date').copy()

fig = go.Figure()
fig.add_trace(go.Scatter(x=ex['date'], y=ex['daily_return'], mode='lines', name=amfi_to_name.get(int(example_code), str(example_code))))
fig.update_layout(
    title=f'Daily Returns Time Series - Example Scheme {example_code}',
    template='plotly_white',
    xaxis_title='Date',
    yaxis_title='daily_return (decimal)',
)
fig.show()

## 2) CAGR (1yr, 3yr, 5yr) + comparison table

In [11]:
# CAGR = (NAV_end / NAV_start)^(1/n) - 1

END_DATE = nav_df['date'].max()
HORIZONS_YEARS = [1, 3, 5]
DAYS_PER_YEAR = 365.25
HORIZON_DELTAS = {y: pd.Timedelta(days=int(round(y * DAYS_PER_YEAR))) for y in HORIZONS_YEARS}

print('END_DATE:', END_DATE)
print('HORIZON_DELTAS:', HORIZON_DELTAS)

END_DATE: 2026-05-29 00:00:00
HORIZON_DELTAS: {1: Timedelta('365 days 00:00:00'), 3: Timedelta('1096 days 00:00:00'), 5: Timedelta('1826 days 00:00:00')}


In [12]:
nav_df = nav_df.sort_values(['amfi_code', 'date']).copy()
groups = dict(tuple(nav_df.groupby('amfi_code', sort=False)))

rows = []

for amfi_code in codes_sorted:
    df = groups.get(amfi_code)
    if df is None or df.empty:
        continue

    df = df[['date', 'nav']].copy().sort_values('date')

    # NAV_end at END_DATE (fallback to closest earlier)
    end_rows = df.loc[df['date'] == END_DATE, 'nav']
    if end_rows.empty:
        end_rows = df.loc[df['date'] <= END_DATE, 'nav']
        if end_rows.empty:
            continue
        end_date_used = df.loc[df['date'] <= END_DATE, 'date'].iloc[-1]
        end_nav = float(end_rows.iloc[-1])
    else:
        end_date_used = END_DATE
        end_nav = float(end_rows.iloc[0])

    out = {
        'amfi_code': int(amfi_code),
        'scheme_name': amfi_to_name.get(int(amfi_code), str(amfi_code)),
        'nav_end_date': str(end_date_used.date()),
    }

    for y in HORIZONS_YEARS:
        target = END_DATE - HORIZON_DELTAS[y]
        start_slice = df.loc[df['date'] <= target]
        if start_slice.empty:
            out[f'cagr_{y}yr_pct'] = np.nan
            continue

        start_nav = float(start_slice['nav'].iloc[-1])
        n_years = float(y)
        cagr = (end_nav / start_nav) ** (1.0 / n_years) - 1.0
        out[f'cagr_{y}yr_pct'] = cagr * 100.0

    rows.append(out)

comparison_df = pd.DataFrame(rows)

# Sort by 5yr CAGR
comparison_table = comparison_df.sort_values('cagr_5yr_pct', ascending=False).reset_index(drop=True)

out_path = DATA_DIR / 'cagr_comparison_1yr_3yr_5yr.csv'
comparison_table.to_csv(out_path, index=False)

print('Wrote:', out_path.resolve())
comparison_table.head(10)

Wrote: C:\Mutual Fund Analytics\Data\processed\cagr_comparison_1yr_3yr_5yr.csv


,amfi_code,scheme_name,nav_end_date,cagr_1yr_pct,cagr_3yr_pct,cagr_5yr_pct
0,100016,HDFC Top 100 Fund - Regular Plan - Growth,2026-05-29,-2.224271,1.292649,NaN
1,100025,HDFC Short Term Debt Fund - Regular - Growth,2026-05-29,3.704969,3.916390,NaN
2,100033,HDFC Mid-Cap Opportunities Fund - Regular - Gr...,2026-05-29,53.232396,32.442459,NaN
3,101206,ABSL Frontline Equity Fund - Regular - Growth,2026-05-29,47.924120,28.967695,NaN
4,101207,ABSL Small Cap Fund - Regular - Growth,2026-05-29,-23.986032,-4.152381,NaN
5,101208,ABSL Liquid Fund - Regular - Growth,2026-05-29,7.236645,6.315784,NaN
6,102885,UTI Nifty 50 Index Fund - Regular - Growth,2026-05-29,20.207704,19.667262,NaN
7,102886,UTI Mid Cap Fund - Regular - Growth,2026-05-29,-16.797481,-0.767406,NaN
8,102887,UTI Flexi Cap Fund - Regular - Growth,2026-05-29,13.583135,25.556188,NaN
9,118632,Nippon India Large Cap Fund - Regular - Growth,2026-05-29,33.981048,22.652360,NaN
